제거 대상 피처 목록과 이유

1. user_no, product_cd, reg_date, end_date
팀원 컬럼 설명에도 "모델링 시 제거" 라고 명시되어 있습니다. user_no와 product_cd는 식별자이고, reg_date와 end_date는 날짜 원본값으로 이미 duration_days, reg_weekday, is_night_signup 등으로 파생이 완료되었습니다. 원본을 남겨두면 모델이 날짜 숫자 자체를 학습하는 오류가 생깁니다.

2. is_same_day_cancel 또는 duration_days 중 하나
둘의 관계를 확인하겠습니다.Is_same_day_cancel과 duration_days 관계 확인Is_same_day_cancel과 duration_days 관계 확인is_same_day_cancel=1이면 duration_days=0이 196건, is_same_day_cancel=0이면 duration_days=0이 30건 존재합니다. 완전한 일치는 아니지만 거의 같은 정보입니다. duration_days가 이진화된 변수로서 더 넓은 정보(30일 미만 전체)를 담고 있으므로, is_same_day_cancel을 제거하고 duration_days를 유지하는 것이 맞습니다. 당일 해지는 30일 미만의 부분집합이므로 duration_days=0이 이미 이 신호를 흡수합니다.

3. promotion_yn 또는 is_promotional_price 중 하나Promotion_yn과 is_promotional_price 관계 확인Promotion_yn과 is_promotional_price 관계 확인완전히 동일한 변수입니다. 두 컬럼이 1:1로 일치합니다. 컬럼 설명을 보면 promotion_yn은 "프로모션 참여 여부", is_promotional_price는 "100원 프로모션 결제 여부"로 정의가 미묘하게 다르지만, 실제 데이터에서는 완전히 같습니다. 둘 중 의미가 더 명확한 is_promotional_price를 유지하고 promotion_yn을 제거합니다. 100원 결제라는 구체적 조건이 더 명확한 정의이기 때문입니다.

4. 시청 행동 피처 전체
total_watch_count, total_watch_duration, unique_movies, avg_duration, watch_days_count, has_watch_history, genre_* 10개, recency, weekday_watch_ratio, binge_score, completion_rate, 총 21개입니다.
앞서 확인했듯 이탈과 잔존 사이의 평균 차이가 거의 0에 가깝습니다. 그런데 단순히 "설명력이 낮다"는 것만으로 제거를 주장하기에는 근거가 부족합니다. 더 근본적인 문제가 있습니다.
View_History의 관찰 기간은 2021년 3월 1일~4월 5일입니다. Membership의 reg_date를 보면 가입일이 같은 기간 안에 분포해 있습니다. 즉 어떤 유저는 가입 직후의 시청 기록을, 어떤 유저는 구독 말기의 시청 기록을 캡처한 것입니다. 구독 생애주기의 어느 시점을 보느냐에 따라 시청 패턴은 완전히 달라지는데, 이것이 통제되지 않은 상태에서 집계된 수치는 개인 간 비교가 불가능합니다. 피처 자체가 오염된 상태입니다.
따라서 이 21개는 단순히 설명력이 낮아서가 아니라, 설계 자체에 문제가 있어서 제거해야 합니다. 올바르게 만들려면 View_History를 각 유저의 reg_date~end_date 구간으로 필터링한 뒤 집계해야 합니다. 이 작업이 선행되지 않으면 어떤 형태의 시청 피처도 신뢰할 수 없습니다.


제거 목록 최종 요약

제거 대상이유user_no, product_cd, reg_date, end_date식별자 또는 파생 완료된 원본is_same_day_cancelduration_days와 거의 동일한 정보promotion_ynis_promotional_price와 완전히 동일한 정보시청 행동 피처 21개 전체View_History 집계 구간이 통제되지 않아 피처 설계 자체가 오염됨
총 29개 컬럼이 제거 대상입니다. 남는 피처는 billing_method, concurrent_streams, is_churn_prevented, repurchase, payment_device, is_user_verified, gender, age, duration_days, plan_tier, currency_type, is_promotional_price, is_night_signup, reg_weekday, amt_100, amt_7900, amt_10900, amt_13900, 총 18개입니다.
이 목록에 동의하시면 코드 작성하겠습니다.일단 그 계획을 유지한 채, 이를 뒷받침해주고 새로이 만들 파생변수도 함께 작성하는게 낫지 않겠어? 어떤 파생변수를 만들 수 있을지, 그리고 만들어야 좋을지도 한 번 생각해보겠어?오후 9:12파생변수 설계와 뷰 히스토리 재결합 가능성 검토했음파생변수 설계와 뷰 히스토리 재결합 가능성 검토했음좋습니다. 현재 남은 18개 피처만으로는 설명력에 한계가 있습니다. 무엇을 만들 수 있는지, 그리고 만들 가치가 있는지를 세 가지 출처로 나눠서 생각해보겠습니다.



Membership_v2 내부 컬럼만으로 만들 수 있는 파생변수


billing_method 그룹화
현재 숫자 코드 그대로 남아 있는데, 컬럼 설명에 따르면 의미 있는 범주가 있습니다. 13x=국내카드, 140=iOS, 151=간편결제, 18x=모바일 이렇게 4개 그룹으로 묶으면 숫자의 크기 관계가 없는 명목형 변수로 올바르게 다룰 수 있습니다. 현재 상태로 모델에 넣으면 190 > 140 > 131처럼 크기 관계가 학습되는 오류가 생깁니다.
age_group
age를 10대, 20대, 30대, 40대, 50대 이상으로 구간화합니다. 나이와 이탈 관계가 선형이 아닐 가능성이 높기 때문에, 연속형으로 두는 것보다 구간화가 더 안정적입니다.
unverified_promo (미인증×프로모션 교호작용)
is_user_verified=0이면서 is_promotional_price=1인 집단입니다. 본인 인증도 안 하고 100원 프로모션으로 가입한 사람이라는 의미인데, 이 집단의 이탈률이 특히 높을 것이라는 가설이 성립합니다. 두 변수가 개별로 존재해도 교호작용이 따로 만들어지지 않으면 모델이 이 조합을 직접 포착하기 어렵습니다. 특히 트리 계열이 아닌 선형 모델에서 중요합니다.
amt_ 컬럼 정리*
amt_100은 is_promotional_price와 동일한 정보입니다. 제거 대상입니다. 나머지 amt_7900, amt_10900, amt_13900은 KRW 가격 티어를 원핫인코딩한 것인데, plan_tier와 currency_type이 이미 있다면 이 세 개도 중복일 수 있습니다.분석했던 변수들의 중복성과 정보 가치를 재평가했습니다분석했던 변수들의 중복성과 정보 가치를 재평가했습니다plan_tier만으로는 가격 정보가 완전히 설명되지 않습니다. basic 플랜 안에서도 amt_7900인 경우와 아닌 경우가 반반입니다. 즉 amt_7900, amt_10900, amt_13900은 plan_tier와 독립적인 정보를 가집니다. amt_100만 제거하고 나머지 세 개는 유지해야 합니다.



View_History를 올바르게 재집계해서 만들 수 있는 파생변수


현재 시청 행동 피처들의 문제는 View_History 전체를 집계했기 때문입니다. 각 유저의 reg_date~end_date 구간 안에 해당하는 시청 기록만 필터링해서 집계하면 의미 있는 피처가 됩니다. User_Mapping을 통해 user_no → USER_ID로 연결한 뒤 구간 필터링을 해야 합니다.
이렇게 만들 수 있는 피처들입니다.
first_watch_offset : reg_date로부터 첫 번째 시청까지 걸린 일수입니다. 가입 후 빨리 시청을 시작한 사람일수록 잔존율이 높을 것이라는 가설이 성립합니다. "초기 몰입도" 신호입니다.
watch_density : watch_days_in_period / duration_days_raw. 구독 기간 대비 실제 시청한 날의 비율입니다. 30일 구독했는데 28일을 시청한 사람과 3일만 시청한 사람을 구분합니다. 현재 watch_days_count는 절대값이라 구독 기간이 다른 사람들 간 비교가 안 됩니다.
last_watch_offset : 마지막 시청일로부터 end_date까지의 일수입니다. 구독 종료 직전까지 시청했는지, 아니면 중반 이후 시청을 끊었는지를 보여줍니다. 현재 recency는 마지막 시청일로부터 View_History 데이터 기준 경과일인데, 구독 만료일 기준으로 재정의하는 것이 더 의미 있습니다.
has_watch_in_period : 구독 기간 내에 시청 기록이 한 건이라도 있는지 여부입니다. 현재 has_watch_history는 전체 View_History 기준이라 구독 기간과 무관할 수 있습니다.



만들지 않는 게 나은 것


reg_weekday와 is_night_signup의 교호작용, 장르 선호도 기반 세분화 등은 현재 데이터 규모와 노이즈 수준을 고려할 때 과도한 피처 엔지니어링이 될 수 있습니다. 설명력이 약한 피처를 조합해서 새 피처를 만들어봤자 노이즈가 증폭될 뿐입니다.


결론적으로 추가할 파생변수 목록

파생변수출처핵심 이유billing_method_groupMembership_v2 내부숫자 코드를 명목형으로 올바르게 변환age_groupMembership_v2 내부비선형 관계 포착unverified_promoMembership_v2 내부고위험 집단 교호작용amt_100 제거Membership_v2 내부is_promotional_price와 완전 중복first_watch_offsetView_History 재집계초기 몰입도 신호watch_densityView_History 재집계구독 기간 대비 정규화된 시청 밀도last_watch_offsetView_History 재집계구독 말기 이탈 예고 신호has_watch_in_periodView_History 재집계구독 기간 내 실제 시청 여부
View_History 재집계 피처 4개는 User_Mapping 조인이 필요하고 작업량이 있습니다. 이것까지 이번 코드에 포함할지, 아니면 Membership_v2 내부 변환만 먼저 할지 어떻게 하실 건가요?한 번에 모든 작업을 하는 코드를 선호함. 먼저, 지우는 변수, 만드는 변수 등을 모조리 작성한 뒤에 코드를 제공해줘. 아 그런데, billing method가 13x면 국내카드, 140이면 ios 등의 정보는 '추정'의 정보라서 삭제하고싶은데. 이를 반영한다면?오후 9:15변수 제거 및 생성 계획을 수립하고 코드 작성을 준비했다변수 제거 및 생성 계획을 수립하고 코드 작성을 준비했다billing_method 그룹핑 자체를 없애고 컬럼 전체를 제거하면 됩니다. 그러면 계획이 더 깔끔해집니다. 전체 작업 목록을 먼저 정리하겠습니다.


제거할 컬럼 (총 30개)

컬럼이유user_no, product_cd식별자, 모델 입력 불가reg_date, end_date파생 완료된 원본. 단, 코드 내에서 파생변수 계산에 임시 사용 후 제거billing_method그룹핑 근거가 추정이므로 전체 제거is_same_day_cancelduration_days와 거의 동일한 정보promotion_ynis_promotional_price와 완전 동일amt_100is_promotional_price와 완전 동일시청 행동 피처 21개 전체View_History 집계 구간이 통제되지 않아 피처 설계 오염. 아래에서 올바르게 재생성
시청 행동 피처 21개는 total_watch_count, total_watch_duration, unique_movies, avg_duration, watch_days_count, has_watch_history, genre_드라마, genre_액션, genre_애니메이션, genre_코미디, genre_로맨스, genre_스릴러, genre_SF, genre_어드벤처, genre_판타지, genre_다큐, recency, weekday_watch_ratio, binge_score, completion_rate입니다.


새로 만들 컬럼 (총 6개)

컬럼계산 방법이유age_groupage를 10단위로 구간화. 10대=1, 20대=2, ..., 50대 이상=5나이와 이탈의 관계가 비선형일 가능성unverified_promois_user_verified==0 AND is_promotional_price==1 → 1, 나머지 → 0고위험 집단 교호작용has_watch_in_period구독 기간(reg_date~end_date) 내 View_History 기록 존재 여부구독 기간 내 실제 시청 여부first_watch_offsetreg_date로부터 첫 시청일까지 일수. 시청 없으면 duration_days_raw로 대체초기 몰입도 신호watch_density구독 기간 내 시청 일수 / duration_days_raw. 시청 없으면 0구독 기간 대비 정규화된 시청 밀도last_watch_offsetend_date로부터 마지막 시청일까지 역산한 일수. 시청 없으면 duration_days_raw로 대체구독 말기 이탈 예고 신호
duration_days_raw는 (end_date - reg_date).days로 내부 계산에만 사용하고 최종 컬럼에는 포함하지 않습니다. duration_days(이진화된 버전)는 유지합니다.
View_History 조인 경로는 Membership_v2.user_no → User_Mapping.uid → User_Mapping.USER_ID → View_History.USER_ID이며, View_History.WATCH_DAY를 각 유저의 reg_date~end_date로 필터링합니다.


최종 잔존 컬럼 (총 17개)

concurrent_streams, is_churn_prevented, repurchase, payment_device, is_user_verified, gender, age, duration_days, plan_tier, currency_type, is_promotional_price, is_night_signup, reg_weekday, amt_7900, amt_10900, amt_13900, 그리고 새로 만드는 age_group, unverified_promo, has_watch_in_period, first_watch_offset, watch_density, last_watch_offset까지 총 22개입니다.

In [6]:
import pandas as pd
import numpy as np
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                              precision_score, classification_report)
import xgboost as xgb
import lightgbm as lgb

In [ ]:
INPUT_PATH  = r"Membership_v3.xlsx"
OUTPUT_PATH = r"Membership_v3(all_model_results).xlsx"

In [8]:
df = pd.read_excel(INPUT_PATH)

df["plan_tier_enc"] = df["plan_tier"].map(
    {"기타": 0, "basic": 1, "standard": 2, "premium": 3}
)
df["currency_enc"] = (df["currency_type"] == "USD").astype(int)

device_dummies = pd.get_dummies(df["payment_device"], prefix="dev", drop_first=False)
df = pd.concat([df, device_dummies], axis=1)

In [9]:
BASE_FEATURES = [
    "concurrent_streams", "is_user_verified", "gender", "age",
    "duration_days", "is_promotional_price", "is_night_signup",
    "reg_weekday", "amt_7900", "amt_10900", "amt_13900",
    "age_group", "has_watch_in_period", "first_watch_offset",
    "watch_density", "last_watch_offset",
    "plan_tier_enc", "currency_enc"
] + list(device_dummies.columns)

CASE_A = BASE_FEATURES + ["is_churn_prevented"]  # is_churn_prevented 포함
CASE_B = BASE_FEATURES                            # is_churn_prevented 제외

TARGET = "repurchase"
y      = df[TARGET]
spw    = float(y.value_counts()[0] / y.value_counts()[1])  # 클래스 불균형 보정값

In [10]:
RANDOM_STATE = 42
TEST_SIZE    = 0.2
SKF          = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [11]:
def get_default_models(spw):
    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "DecisionTree": DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_STATE
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100, class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_STATE
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=100, random_state=RANDOM_STATE
        ),
        "XGBoost": xgb.XGBClassifier(
            scale_pos_weight=spw, eval_metric="logloss",
            random_state=RANDOM_STATE, verbosity=0
        ),
        "LightGBM": lgb.LGBMClassifier(
            class_weight="balanced", random_state=RANDOM_STATE, verbose=-1
        ),
    }

In [12]:
def evaluate(model, X_te, y_te):
    prob = model.predict_proba(X_te)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        "AUC":       roc_auc_score(y_te, prob),
        "F1":        f1_score(y_te, pred),
        "Recall":    recall_score(y_te, pred),
        "Precision": precision_score(y_te, pred),
        "Accuracy":  (pred == y_te).mean(),
    }

In [13]:
# ── Step 1. 기본 파라미터 전체 모델 비교 ──────────────────
print("=" * 72)
print("  Step 1. 기본 파라미터 비교 (test set)")
print("=" * 72)
print(f"  {'모델':<22} {'Case':<7} {'AUC':>8} {'F1':>7} {'Recall':>8} {'Prec':>8}")
print("=" * 72)

step1_rows = []
for case_name, features in [("A", CASE_A), ("B", CASE_B)]:
    X      = df[features].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    scaler = StandardScaler()
    X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns)
    X_te_s = pd.DataFrame(scaler.transform(X_te),     columns=X_te.columns)

    for mname, model in get_default_models(spw).items():
        Xi_tr = X_tr_s if mname == "LogisticRegression" else X_tr
        Xi_te = X_te_s if mname == "LogisticRegression" else X_te
        model.fit(Xi_tr, y_tr)
        m = evaluate(model, Xi_te, y_te)
        row = {"Case": case_name, "Model": mname, "Tuned": False, **m}
        step1_rows.append(row)
        print(f"  {mname:<22} Case{case_name:<6} "
              f"{m['AUC']:>8.4f} {m['F1']:>7.4f} "
              f"{m['Recall']:>8.4f} {m['Precision']:>8.4f}")

print("=" * 72)

  Step 1. 기본 파라미터 비교 (test set)
  모델                     Case         AUC      F1   Recall     Prec
  LogisticRegression     CaseA        0.6318  0.6735   0.6186   0.7391
  DecisionTree           CaseA        0.5579  0.6924   0.6821   0.7030
  RandomForest           CaseA        0.6099  0.7651   0.8475   0.6973
  GradientBoosting       CaseA        0.6597  0.8026   0.9420   0.6992
  XGBoost                CaseA        0.6289  0.6963   0.6602   0.7366


  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_c

  LightGBM               CaseA        0.6457  0.7023   0.6619   0.7480
  LogisticRegression     CaseB        0.6297  0.6740   0.6186   0.7402
  DecisionTree           CaseB        0.5543  0.6935   0.6863   0.7010
  RandomForest           CaseB        0.6025  0.7563   0.8295   0.6949
  GradientBoosting       CaseB        0.6512  0.8043   0.9530   0.6958
  XGBoost                CaseB        0.6216  0.6964   0.6627   0.7336
  LightGBM               CaseB        0.6399  0.6966   0.6548   0.7442


In [14]:
# ── Step 2. Optuna 하이퍼파라미터 최적화 ──────────────────
# 대상 모델: XGBoost, LightGBM, GradientBoosting
# n_trials 조정 가능 (높을수록 정밀, 느림)
N_TRIALS = 50

print(f"\n{'='*72}")
print(f"  Step 2. Optuna 최적화 ({N_TRIALS} trials/모델)")
print(f"{'='*72}")

step2_rows = []

for case_name, features in [("A", CASE_A), ("B", CASE_B)]:
    X = df[features].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    print(f"\n  [Case {case_name}]")

    # ── XGBoost ──────────────────────────────────────────
    def obj_xgb(trial):
        p = dict(
            n_estimators     = trial.suggest_int("n_estimators", 100, 500),
            max_depth        = trial.suggest_int("max_depth", 2, 8),
            learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            subsample        = trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_weight = trial.suggest_int("min_child_weight", 1, 10),
            scale_pos_weight = spw,
            eval_metric="logloss", random_state=RANDOM_STATE, verbosity=0
        )
        return cross_val_score(
            xgb.XGBClassifier(**p), X_tr, y_tr,
            cv=SKF, scoring="roc_auc", n_jobs=-1
        ).mean()

    study_xgb = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study_xgb.optimize(obj_xgb, n_trials=N_TRIALS, show_progress_bar=False)
    bp = study_xgb.best_trial.params
    best_xgb = xgb.XGBClassifier(
        n_estimators=bp["n_estimators"], max_depth=bp["max_depth"],
        learning_rate=bp["learning_rate"], subsample=bp["subsample"],
        colsample_bytree=bp["colsample_bytree"], min_child_weight=bp["min_child_weight"],
        scale_pos_weight=spw, eval_metric="logloss",
        random_state=RANDOM_STATE, verbosity=0
    )
    best_xgb.fit(X_tr, y_tr)
    m = evaluate(best_xgb, X_te, y_te)
    row = {"Case": case_name, "Model": "XGBoost(Optuna)", "Tuned": True,
           "CV_AUC": study_xgb.best_value, "Best_Params": str(bp), **m}
    step2_rows.append(row)
    print(f"    XGBoost      CV_AUC={study_xgb.best_value:.4f}  "
          f"Test_AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}")

    # ── LightGBM ─────────────────────────────────────────
    def obj_lgb(trial):
        p = dict(
            n_estimators     = trial.suggest_int("n_estimators", 100, 500),
            max_depth        = trial.suggest_int("max_depth", 2, 8),
            learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            num_leaves       = trial.suggest_int("num_leaves", 20, 100),
            subsample        = trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_samples= trial.suggest_int("min_child_samples", 10, 50),
            class_weight="balanced", random_state=RANDOM_STATE, verbose=-1
        )
        return cross_val_score(
            lgb.LGBMClassifier(**p), X_tr, y_tr,
            cv=SKF, scoring="roc_auc", n_jobs=-1
        ).mean()

    study_lgb = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study_lgb.optimize(obj_lgb, n_trials=N_TRIALS, show_progress_bar=False)
    blp = study_lgb.best_trial.params
    best_lgb = lgb.LGBMClassifier(
        n_estimators=blp["n_estimators"], max_depth=blp["max_depth"],
        learning_rate=blp["learning_rate"], num_leaves=blp["num_leaves"],
        subsample=blp["subsample"], colsample_bytree=blp["colsample_bytree"],
        min_child_samples=blp["min_child_samples"],
        class_weight="balanced", random_state=RANDOM_STATE, verbose=-1
    )
    best_lgb.fit(X_tr, y_tr)
    m = evaluate(best_lgb, X_te, y_te)
    row = {"Case": case_name, "Model": "LightGBM(Optuna)", "Tuned": True,
           "CV_AUC": study_lgb.best_value, "Best_Params": str(blp), **m}
    step2_rows.append(row)
    print(f"    LightGBM     CV_AUC={study_lgb.best_value:.4f}  "
          f"Test_AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}")

    # ── GradientBoosting ──────────────────────────────────
    def obj_gb(trial):
        p = dict(
            n_estimators    = trial.suggest_int("n_estimators", 100, 500),
            max_depth       = trial.suggest_int("max_depth", 2, 6),
            learning_rate   = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            subsample       = trial.suggest_float("subsample", 0.6, 1.0),
            min_samples_leaf= trial.suggest_int("min_samples_leaf", 10, 50),
            random_state    = RANDOM_STATE
        )
        return cross_val_score(
            GradientBoostingClassifier(**p), X_tr, y_tr,
            cv=SKF, scoring="roc_auc", n_jobs=-1
        ).mean()

    study_gb = optuna.create_study(
        direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study_gb.optimize(obj_gb, n_trials=N_TRIALS, show_progress_bar=False)
    bgp = study_gb.best_trial.params
    best_gb = GradientBoostingClassifier(**bgp)
    best_gb.fit(X_tr, y_tr)
    m = evaluate(best_gb, X_te, y_te)
    row = {"Case": case_name, "Model": "GradientBoosting(Optuna)", "Tuned": True,
           "CV_AUC": study_gb.best_value, "Best_Params": str(bgp), **m}
    step2_rows.append(row)
    print(f"    GradBoosting CV_AUC={study_gb.best_value:.4f}  "
          f"Test_AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}")


  Step 2. Optuna 최적화 (50 trials/모델)

  [Case A]
    XGBoost      CV_AUC=0.6628  Test_AUC=0.6577  Recall=0.6766
    LightGBM     CV_AUC=0.6619  Test_AUC=0.6572  Recall=0.6737
    GradBoosting CV_AUC=0.6624  Test_AUC=0.6559  Recall=0.9811

  [Case B]
    XGBoost      CV_AUC=0.6557  Test_AUC=0.6480  Recall=0.6644
    LightGBM     CV_AUC=0.6563  Test_AUC=0.6505  Recall=0.6678
    GradBoosting CV_AUC=0.6555  Test_AUC=0.6506  Recall=0.9567


In [15]:
all_rows = step1_rows + step2_rows
res = pd.DataFrame(all_rows)

print(f"\n{'='*72}")
print("  최종 결과 요약 (AUC 내림차순)")
print(f"{'='*72}")
print(f"  {'모델':<28} {'Case':<7} {'AUC':>8} {'F1':>7} {'Recall':>8} {'Prec':>8}")
print(f"{'='*72}")
for case in ["A", "B"]:
    sub = res[res["Case"] == case].sort_values("AUC", ascending=False)
    for _, r in sub.iterrows():
        mark = " ★" if r["AUC"] == sub["AUC"].max() else ""
        print(f"  {r['Model']:<28} Case{r['Case']:<6} "
              f"{r['AUC']:>8.4f} {r['F1']:>7.4f} "
              f"{r['Recall']:>8.4f} {r['Precision']:>8.4f}{mark}")
    print(f"  {'-'*68}")

best_a = res[res["Case"] == "A"].sort_values("AUC", ascending=False).iloc[0]
best_b = res[res["Case"] == "B"].sort_values("AUC", ascending=False).iloc[0]
print(f"\n  CaseA 최고: {best_a['Model']:<28} AUC={best_a['AUC']:.4f}  Recall={best_a['Recall']:.4f}")
print(f"  CaseB 최고: {best_b['Model']:<28} AUC={best_b['AUC']:.4f}  Recall={best_b['Recall']:.4f}")
print(f"  AUC 차이 (A-B): {best_a['AUC'] - best_b['AUC']:+.4f}")

res.to_excel(OUTPUT_PATH, index=False)
print(f"\n저장 완료: {OUTPUT_PATH}")


  최종 결과 요약 (AUC 내림차순)
  모델                           Case         AUC      F1   Recall     Prec
  GradientBoosting             CaseA        0.6597  0.8026   0.9420   0.6992 ★
  XGBoost(Optuna)              CaseA        0.6577  0.7114   0.6766   0.7500
  LightGBM(Optuna)             CaseA        0.6572  0.7100   0.6737   0.7506
  GradientBoosting(Optuna)     CaseA        0.6559  0.8080   0.9811   0.6869
  LightGBM                     CaseA        0.6457  0.7023   0.6619   0.7480
  LogisticRegression           CaseA        0.6318  0.6735   0.6186   0.7391
  XGBoost                      CaseA        0.6289  0.6963   0.6602   0.7366
  RandomForest                 CaseA        0.6099  0.7651   0.8475   0.6973
  DecisionTree                 CaseA        0.5579  0.6924   0.6821   0.7030
  --------------------------------------------------------------------
  GradientBoosting             CaseB        0.6512  0.8043   0.9530   0.6958 ★
  GradientBoosting(Optuna)     CaseB        0.6506  0.8059